In [ ]:
!pip install matplotlib==3.10.6 \
numpy==2.3.3\
pandas==2.3.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 103.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sour

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
tables = []

for year in range(2013, 2026):
  tables.append(pd.read_excel(f"SPK_IPOs_{year}.xlsx"))

all_ipos = pd.concat(tables, ignore_index=True)
print(f"Dataset: {all_ipos.shape[0]} IPOs, {all_ipos.shape[1]} columns")
print(all_ipos.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: 'SPK_IPOs_2013.xlsx'

In [ ]:
tables_overs = []

for year in range(2020, 2024):
  tables_overs.append(pd.read_excel(f"SPK_IPOs_Overs_{year}.xlsx", header=1))

all_ipos_overs = pd.concat(tables_overs, ignore_index=True)
print(f"Dataset overs: {all_ipos_overs.shape[0]} IPOs, {all_ipos_overs.shape[1]} columns")
print(all_ipos_overs.columns.tolist())

In [ ]:
print("MAIN")
print(all_ipos.head())
print()
print("OVERS")
print(all_ipos_overs.head())

Clean the data and format it so that we can merge them correctly


In [ ]:
all_ipos["Borsada İşlem Görme Tarihi"] = pd.to_datetime(
    all_ipos["Borsada İşlem Görme Tarihi"], format="%d.%m.%Y"
)

print(f"Rows before cleaning: {len(all_ipos)}")
print(f"Rows with missing ticker: {all_ipos['Borsa Kodu'].isna().sum()}")
print(f"Rows with missing listing date: {all_ipos['Borsada İşlem Görme Tarihi'].isna().sum()}")

In [ ]:
all_ipos = all_ipos.dropna(subset=["Borsa Kodu"])
print(f"Rows after cleaning: {len(all_ipos)}")

In [ ]:
# Remove TOPLAM rows and drop Sıra column
all_ipos_overs = all_ipos_overs[all_ipos_overs["Şirket Adı"] != "TOPLAM"]
all_ipos_overs = all_ipos_overs.dropna(subset=["Şirket Adı"])
all_ipos_overs = all_ipos_overs.drop(columns=["Sıra              "])
print(f"Overs after cleaning: {len(all_ipos_overs)}")

In [ ]:
print("=== Main columns ===")
print(all_ipos[["Borsa Kodu", "Şirket Ünvanı", "Halka Arz Oranı (%)",
                "Halka Arz Fiyatı (TL)", "Yeni Sermaye (Bin TL)",
                "Halka Arza Aracılık Eden Kurum",
                "Borsada İşlem Görme Tarihi"]].head())

In [ ]:
kap_sectors = pd.read_excel("KAP_Sectors.xlsx")
print(f"Sectors: {kap_sectors.shape}")
print(kap_sectors.columns.tolist())
print()
print(kap_sectors.head())

In [ ]:
# Parse KAP sector data
raw = pd.read_excel("KAP_Sectors.xlsx")
cols = raw.columns

sector_map = {}
current_sector = None

for _, row in raw.iterrows():
    val0 = str(row[cols[0]]).strip() if pd.notna(row[cols[0]]) else None
    val1 = row[cols[1]]

    # Skip empty rows, "Sıra" headers, "Kayıt Bulunamadı"
    if val0 is None or val0 == "nan":
        continue
    if val0 == "Sıra" or val0 == "Kayıt Bulunamadı":
        continue

    # If second column is empty, this row is a sector name
    if pd.isna(val1):
        current_sector = val0
    else:
        # This is a company row — val1 is the ticker
        ticker = str(val1).strip()
        sector_map[ticker] = current_sector

kap_df = pd.DataFrame(list(sector_map.items()), columns=["Borsa Kodu", "Sektör"])
print(f"Sectors mapped: {len(kap_df)} companies")
print(kap_df.head(10))

In [ ]:
all_ipos = all_ipos.merge(kap_df, on="Borsa Kodu", how="left")

print(f"IPOs with sector: {all_ipos['Sektör'].notna().sum()}")
print(f"IPOs without sector: {all_ipos['Sektör'].isna().sum()}")

In [ ]:
print(all_ipos[all_ipos["Sektör"].isna()][["Borsa Kodu", "Şirket Ünvanı"]])

In [ ]:
ticker_fixes = {
    "EFORC": "EFOR",
    "SNKRN": "GATEG",
    "KARYE": "A1YEN",
    "GRTRK": "GRTHO",
}

delisted = ["AKPAZ", "ARBUL", "VIAGO"]

for old, new in ticker_fixes.items():
    all_ipos.loc[all_ipos["Borsa Kodu"] == old, "Borsa Kodu"] = new
    match = kap_df[kap_df["Borsa Kodu"] == new]["Sektör"]
    if len(match) > 0:
        all_ipos.loc[all_ipos["Borsa Kodu"] == new, "Sektör"] = match.values[0]

# Check remaining missing sectors
missing = all_ipos[all_ipos["Sektör"].isna()][["Borsa Kodu", "Şirket Ünvanı"]]
print(f"Still missing sector: {len(missing)}")
print(missing)

In [ ]:
manual_sectors = {
    "OYYAT": "ARACI KURUMLAR",
    "TERA": "ARACI KURUMLAR",
    "A1CAP": "ARACI KURUMLAR",
    "SKYMD": "ARACI KURUMLAR",
    "VIAGO": "GAYRİMENKUL YATIRIM ORTAKLIKLARI",
    "AKPAZ": "TOPTAN VE PERAKENDE TİCARET",
    "ARBUL": "TEKSTİL, GİYİM EŞYASI VE DERİ",
}

for ticker, sector in manual_sectors.items():
    all_ipos.loc[all_ipos["Borsa Kodu"] == ticker, "Sektör"] = sector

print(f"Still missing sector: {all_ipos['Sektör'].isna().sum()}")

In [ ]:
all_ipos.to_csv("ipo_main_cleaned.csv", index=False)
all_ipos_overs.to_csv("ipo_overs_cleaned.csv", index=False)
print("Saved!")
print(f"Main: {len(all_ipos)} IPOs, {len(all_ipos.columns)} columns")
print(f"Overs: {len(all_ipos_overs)} IPOs, {len(all_ipos_overs.columns)} columns")

In [ ]:
print(all_ipos.tail(5))
print()
# Check for rows where Borsa Kodu looks like a number or "Toplam" etc.
print(all_ipos[all_ipos["Borsa Kodu"].str.match(r'^\d+$', na=False)])

In [ ]:
print(all_ipos[all_ipos["Halka Arz Fiyatı (TL)"].isna()])

In [ ]:
print(all_ipos.groupby(all_ipos["Borsada İşlem Görme Tarihi"].dt.year).size())